# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
#Instalar librerias
!pip install statsmodels
# importar librerías
import pandas as pd
import numpy as np
from scipy.stats import levene
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# encabezado dataset de orders
orders.head(5)

In [ ]:
# exploración dataset de orders
orders.info()

In [ ]:
# encabezado dataset de catalog
catalog.head(5)

In [ ]:
# exploración dataset de catalog
catalog.info()

In [ ]:
# Encabezado dataset markenting
marketing.head(5)


In [ ]:
# Encabezado dataset markenting
marketing.info()


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [ ]:
# Conversión de fechas dataset orders y marketing
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'],errors="coerce")
marketing['fecha']=pd.to_datetime(marketing['fecha'],errors="coerce")

In [ ]:
#Verificación de variables númericas
columnas_numericas=['cantidad','precio_unitario','monto_descuento','monto_total']
orders[columnas_numericas].describe()



In [ ]:
#Verificación de variables númericas en el dataset catalog
catalog['costo_unitario'].describe()



In [ ]:
#Verificación de variables númericas en el dataset marketing
marketing['gasto'].describe()



In [ ]:
# Verificacion de filas menores que 0 en la columna cantidad
orders[orders['cantidad']<0]

In [ ]:
# eliminación de valores menores que 0 en la columna cantidad
orders= orders[orders['cantidad']>=0]
orders.describe()

In [ ]:
# Conversión de cantidad de float a integer

orders['cantidad']=orders['cantidad'].astype(int)
orders.info()

In [ ]:
# Verificación de consistencia de montos
orders['total_calculado']= orders['cantidad']*orders['precio_unitario']-orders['monto_descuento']
orders['diferencia_montos']=orders['total_calculado']-orders['monto_total']
orders['diferencia_montos'].describe()

**Resultado de verificación consistencia de montos**:
- Ya que los montos presentan bajas diferencias significativas, se procede a hacer la eliminación de estos

<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

Excelente trabajo en esta sección, Nirvana.

Me gustó que no te limitaras a revisar los tipos de datos, sino que también incorporaras una validación para comprobar la consistencia entre el monto registrado y el monto calculado.

La comparación entre <code>cantidad * precio_unitario - monto_descuento</code> y <code>monto_total</code> es una buena práctica.

Además, los resultados de tu validación muestran diferencias muy pequeñas, de hasta 0.01, lo que permite identificar posibles efectos de redondeo.

En un entorno profesional, estas comprobaciones ayudan a garantizar que las métricas financieras se construyan sobre información consistente.

¡Muy buen criterio analítico!

</div>

In [ ]:
# Eliminación de columnas de validación
orders = orders.drop(columns=['total_calculado', 'diferencia_montos'])

In [ ]:
#Verificación de valores duplicados
orders.duplicated(subset=['id_pedido']).sum()

In [ ]:
orders[orders.duplicated(subset=['id_pedido'], keep=False)].sort_values('id_pedido')

In [ ]:
#Eliminación de duplicados
orders = orders.drop_duplicates(subset=['id_pedido']).reset_index()

In [ ]:
#Validación de eliminación de duplicados en el dataset de orders
orders.duplicated(subset=['id_pedido']).sum()

In [ ]:
#Revisión de duplicados en el dataset de catalog
catalog.duplicated(subset=['nombre_producto']).sum()

In [ ]:
#Revisión de duplicados en el dataset de marketing
marketing.duplicated(subset=['fecha', 'id_campaña', 'canal']).sum()

In [ ]:
#Verificación de variables en orders
variables_categoricas_orders=['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto']

for cat in variables_categoricas_orders:
    print(orders[cat].value_counts())
    


In [ ]:
#Cambiar nombre de columnas
orders['pais'] = orders['pais'].str.capitalize()
orders['categoria_producto'] = orders['categoria_producto'].replace('Electronica', 'Electrónica')
#Verificar el cambio realizado
variables= ['pais','categoria_producto']
orders[variables].value_counts()

In [ ]:
#Verificacion variables en catalog
variables_categoricas_catalog=['nombre_producto','categoria_producto','proveedor']

for cat in variables_categoricas_catalog:
    print(catalog[cat].value_counts())

In [ ]:
variables_categoricas_marketing=['pais','canal']

for cat in variables_categoricas_marketing:
    print(marketing[cat].value_counts())

**Observaciones**:
- En orders se encontró en las columnas de **pais** y **categoria_producto** nombres que no estaban en mayúsculas, ni nombres de datos con las respectivas tiles, razón por la cuál se hicieron las respectivas transformaciones.
- En el dataset de catalog, no se presentaron inconvenientes en cuanto los nombres con modificaciones,tampoco, en el dataset de marketing


In [ ]:
#Verificación de nulos en el dataset de orders
orders.isnull().sum()

**Observaciones**:
- Ya que las columnas categoricas presentan valores nulos muy bajos en porcentajes respectivamente:
- **Pais**: 1.18%
- **Dispositivo**:0.08%
- **fuente_referencia**:0.12%
- **nombre_producto**:0.12%
- **categoria_producto**:0.12%

Se procede a hacer la eliminación de estas filas, ya que no son significativas, ni afectarías los respectivos cálculos estadísticos.

In [ ]:
#Eliminacion de nulos
columnas_nulos=['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto']
orders=orders.dropna(subset=columnas_nulos)

In [ ]:
#Verificación de nulos en el dataset de catalog
catalog.isnull().sum()

**Observaciones**:
-Como no se presentaron valores nulos en las columnas de este dataset, no se requieren hacer modificaciones

In [ ]:
#Verificación de nulos en el dataset de marketing
marketing.isnull().sum()

**Observaciones**:
-Como en la columna canal del dataset de marketing presenta el 6.2% de valores nulos, se procede a hacer la imputacion con el nombre desconocido, para no afectar los cálculos de gasto de campaña 

In [ ]:
#Imputacion de nulos a  nombrarlos por desconocido
marketing['canal']=marketing['canal'].fillna("Desconocido")
#Verificación de imputación de nulos a desconocido
marketing.isnull().sum()

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:
# Cálculo de los KPIs principales
ingreso_total=orders['monto_total'].sum()
print(ingreso_total)

In [ ]:
#Para costo total se tienen que unir los datasets de catalog y orders
merge_costo= pd.merge(orders,catalog,on=['nombre_producto'],how='left')
costo_total = (merge_costo['cantidad'] * merge_costo['costo_unitario']).sum()
print(costo_total)

In [ ]:
#Para inversión de marketing
inversion_marketing= marketing['gasto'].sum()
print(inversion_marketing)

In [ ]:
#Cálculo del profit
profit=ingreso_total-(costo_total+inversion_marketing)
print(profit)

In [ ]:
#Cálculo margen de ganancia
margen_ganancia= (profit/ingreso_total)*100.0
print(margen_ganancia)

In [ ]:
# Cálculo ticket promedio
ticket_promedio = ingreso_total / (orders['id_pedido'].count())
print(ticket_promedio)

In [ ]:
#Cantidad promedio
cantidad_promedio = orders['cantidad'].mean()
print(cantidad_promedio)

In [ ]:
#Producto más vendido
cantidad_productos_vendidos=orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print(cantidad_productos_vendidos)

In [ ]:
#Gasto de marketing por canal
gasto_canal=marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print(gasto_canal)

**Observaciones**:
- El **ingreso total** es de 51'836,375.14, el **costo total** es de 43'078,678.8, la **inversión de marketing** es de 2'871,843.53, el **profit** es de 5'885,852.80 y el **margen de ganancia** es de 11.36%.
- El **ticket promedio por orden**  es de 2107.16, la **cantidad promedio de productos por orden** es de 7.2, el **producto mas vendido** es Laptop-Gaming-16GB y el **gasto de marketing por canal** que presento mayor inversión fue de social, seguido de orgánico, además los canales que presentaron menor inversión fue paid_search y desconocido.


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [70]:
#Importar libreria de SQL
!pip install sqlalchemy
!pip install psycopg[binary]
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [72]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS total
FROM events
GROUP BY nombre_evento;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total
0,add_payment_info,6250
1,add_to_cart,7634
2,begin_checkout,7208
3,first_visit,7796
4,purchase,6240
5,select_item,7582


In [74]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),

select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
      AND id_usuario IN (
          SELECT id_usuario FROM first_visit
      )
),

add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
      AND id_usuario IN (
          SELECT id_usuario FROM select_item
      )
),

begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
      AND id_usuario IN (
          SELECT id_usuario FROM add_to_cart
      )
),

add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
      AND id_usuario IN (
          SELECT id_usuario FROM begin_checkout
      )
),

purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
      AND id_usuario IN (
          SELECT id_usuario FROM add_payment_info
      )
)

SELECT
    (SELECT COUNT(*) FROM select_item) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM first_visit), 0)
        AS conversion_to_select_item,

    (SELECT COUNT(*) FROM add_to_cart) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM select_item), 0)
        AS conversion_to_add_to_cart,

    (SELECT COUNT(*) FROM begin_checkout) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM add_to_cart), 0)
        AS conversion_to_begin_checkout,

    (SELECT COUNT(*) FROM add_payment_info) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM begin_checkout), 0)
        AS conversion_to_add_payment_info,

    (SELECT COUNT(*) FROM purchase) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM add_payment_info), 0)
        AS conversion_to_purchase,

    (SELECT COUNT(*) FROM purchase) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM first_visit), 0)
        AS conversion_final,

    ((SELECT COUNT(*) FROM first_visit) -
     (SELECT COUNT(*) FROM select_item)) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM first_visit), 0)
        AS abandono_first_visit,

    ((SELECT COUNT(*) FROM select_item) -
     (SELECT COUNT(*) FROM add_to_cart)) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM select_item), 0)
        AS abandono_select_item,

    ((SELECT COUNT(*) FROM add_to_cart) -
     (SELECT COUNT(*) FROM begin_checkout)) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM add_to_cart), 0)
        AS abandono_add_to_cart,

    ((SELECT COUNT(*) FROM begin_checkout) -
     (SELECT COUNT(*) FROM add_payment_info)) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM begin_checkout), 0)
        AS abandono_begin_checkout,

    ((SELECT COUNT(*) FROM add_payment_info) -
     (SELECT COUNT(*) FROM purchase)) * 100.0 /
        NULLIF((SELECT COUNT(*) FROM add_payment_info), 0)
        AS abandono_add_payment_info;



'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversion_to_select_item,conversion_to_add_to_cart,conversion_to_begin_checkout,conversion_to_add_payment_info,conversion_to_purchase,conversion_final,abandono_first_visit,abandono_select_item,abandono_add_to_cart,abandono_begin_checkout,abandono_add_payment_info
0,94.830682,95.387529,90.243902,78.048397,77.652507,49.474089,5.169318,4.612471,9.756098,21.951603,22.347493


**Observaciones**:

-El funnel presenta las siguientes tasas de conversion:
**First visit a select item**: 94.83%
**Select item a add to cart**: 95.39%
**Add to cart a begin checkout**: 90.24%
**Begin checkout a add payment info**: 78.05%
**Add payment info a purchase**: 77.65%
**Conversion final**: 49.47%

-El funnel presenta las siguientes tasas de abandono:
**First visit a select item**: 5.17%
**Select item a add to cart**: 4.61%
**Add to cart a begin checkout**: 9.76%
**Begin checkout a add payment info**: 21.95%
**Add payment info a purchase**: 22.34%

-Las tasas de abandono en la página fueron bajas, ademas en las etapas que se presentaron mayor abandono fue en las etapas de **Begin checkout** a **add payment info** y en **add payment info** a **purchase**.

**Recomendacion**:
-Se recomienda analizar los posibles factores que incrementen la tasa de abandono en estas etapa, como cambios en los precios, formularios de datos bancarios muy largos, entre otros y fallas en la plataforma al momento de ingresar los datos bancarios

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohort AS (

    -- 1. Identificar la cohorte según el mes de registro
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        TO_CHAR(
            DATE_TRUNC('month', CAST(fecha_registro AS DATE)),
            'YYYY-MM'
        ) AS cohorte_inicial
    FROM users
    GROUP BY
        id_usuario,
        fecha_registro

),

activity AS (

    -- 2. Relacionar usuarios con su actividad
    SELECT
        c.id_usuario,
        c.cohorte_inicial,
        a.dias_despues_registro,
        a.activo
    FROM cohort AS c
    LEFT JOIN user_activity AS a
        ON c.id_usuario = a.id_usuario

)

-- 3. Calcular cantidades y porcentajes directamente
SELECT
    cohorte_inicial,

    -- Usuarios iniciales
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales,

    -- Semana 1
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro>=1 
        AND activo = 1
        THEN id_usuario
    END) AS retenido_w1,

    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN dias_despues_registro>= 1
            AND activo = 1
            THEN id_usuario
        END) / COUNT(DISTINCT id_usuario),
        2
    ) AS semana_1,

    -- Semana 2
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro >7
        AND activo = 1
        THEN id_usuario
    END) AS retenido_w2,

    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN dias_despues_registro > 7
            AND activo = 1
            THEN id_usuario
        END) / COUNT(DISTINCT id_usuario),
        2
    ) AS semana_2,

    -- Semana 3
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro >14
        AND activo = 1
        THEN id_usuario
    END) AS retenido_w3,

    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN dias_despues_registro >14
            AND activo = 1
            THEN id_usuario
        END) / COUNT(DISTINCT id_usuario),
        2
    ) AS semana_3

FROM activity

GROUP BY cohorte_inicial

ORDER BY cohorte_inicial;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

**Insight**:
Se presentó una disminución constante en la retención acumulada en promedio de usuarios de la siguiente forma: usuarios activos después de la semana 1(86.32%), usuarios activos después semana 2(78.72%) y usuarios activos después semana 3(64.21%). Lo cual se nota un patrón de disminucion constante a lo largo de las semanas. Aunque estos valores se muestren altos, hay que revisar, si los usuarios solo abrieron la plataforma y luego cerraron la aplicación y esto se conto como activo o permanecieron mas tiempo utilizandola.

**Recomendacion**:
- Se recomienda analizar  de forma simultanea en las tablas de user_activity y orders, 
si la cantidad de usuarios activos se traducen en beneficios economicos para el negocio



---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión es igual entre grupos, es decir, cualquier diferencia se debe al azar del muestreo
   - **H₁ (Hipótesis alternativa):** La tasa de conversión es diferente entre ambos grupos, es decir, las diferencias no se deben al azar del muestreo 
   
**Test estadístico:** Prueba z  
**Nivel de significancia alpha:** 0.05

In [ ]:
experiment_checkout= pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
conversiones=experiment_checkout.groupby('variante')['convirtio'].sum()
totales= experiment_checkout.groupby('variante')['convirtio'].count()
print("Conversiones por grupo:")
print(conversiones)

print("Totales por grupo:")
print(totales)


In [ ]:
#Ahora se pasan los valores a formato de lista
exitos=[conversiones['control'],conversiones['tratamiento']]
observaciones=[totales['control'],totales['tratamiento']]

In [ ]:
#Aplicación de prueba estadística
z_stat,p_value= proportions_ztest(exitos,observaciones)
print(f"Estadístico : {z_stat}")
print(f"Valor p: {p_value}")

In [ ]:
#Interpretación de resultados

alpha=0.05

if p_value < 0.05:
    print("Rechazamos la hipótesis nula: Hay evidencia de una diferencia")
else:
    print("No rechazamos la hipótesis nula: No hay evidencia de una diferencia")

In [ ]:
#Calculo de tasas de conversión

tasa_control=exitos[0]/observaciones[0]
tasa_tratamiento=exitos[1]/observaciones[1]
print(f"Tasa de conversión del grupo de control:{tasa_control:.2%}")
print(f"Tasa de conversión del grupo de tratamiento:{tasa_tratamiento:.2%}")
#Interpretación de los resultados
if tasa_control > tasa_tratamiento:
    print(f"\n El grupo de control tiene una mayor tasa de conversión({tasa_control-tasa_tratamiento:.2%})")
    
elif tasa_tratamiento>tasa_control:
    print(f"\n El grupo de tratamiento tiene una mayor tasa de conversión({tasa_tratamiento-tasa_control:.2%})")

else:
    print("\n Ambas páginas tienen la misma tasa de conversión")

**Recomendaciones**
- No se recomienda seguir la implementación de cambios en el producto ya que no presentaron diferencias significativas en los grupos al momento de hacer la compra, esto se evidencia en el **valor p** que muestra un valor de 0.416, siendo este mucho mayor al valor p esperado de 0.05. Adicionalmente, la diferencia de **tasas de conversión** entre el grupo de control y el grupo de tratamiento es del 0.6%, presentando un mayor valor el grupo de tratamiento que es del 16.29%.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
https://public.tableau.com/app/profile/nirvana.alzate/viz/Nirvana_Alzate_Proyecto_Final/Overview#1